# DACN AI Runtime serving

Connect VS Code to a Google Colab **Tesla T4** runtime, then use **Run All**. This notebook starts the existing FastAPI runtime and leaves it serving; it does not run inference fixtures or stop the server at the end.

The server binds to Colab's `127.0.0.1:8000`, reachable only inside this Colab runtime. A local NestJS process cannot use that localhost address. Colab may stop the server when its runtime disconnects.

In [ ]:
# Setup: clone or reuse the exact source commit; never follow a moving branch.
import pathlib, re, subprocess


def checkout_source(repo_url, commit, root="/content"):
    if not isinstance(repo_url, str) or not repo_url.strip():
        raise ValueError("Enter a repository URL.")
    if not isinstance(commit, str) or not re.fullmatch(r"[0-9a-f]{40}", commit.lower()):
        raise ValueError("Enter an immutable 40-character commit SHA.")
    commit = commit.lower()
    target = pathlib.Path(root) / f"dacn-ai-serving-{commit[:12]}"
    try:
        if target.exists():
            current = subprocess.check_output(
                ["git", "-C", str(target), "rev-parse", "HEAD"], text=True, stderr=subprocess.DEVNULL
            ).strip()
            if current != commit:
                raise RuntimeError("Existing source directory has a different commit; choose a clean Colab runtime.")
        else:
            subprocess.run(
                ["git", "clone", "--filter=blob:none", "--no-checkout", "--quiet", repo_url, str(target)],
                check=True, capture_output=True, text=True,
            )
            subprocess.run(
                ["git", "-C", str(target), "fetch", "--quiet", "--depth", "1", "origin", commit],
                check=True, capture_output=True, text=True,
            )
            subprocess.run(
                ["git", "-C", str(target), "checkout", "--quiet", "--detach", commit],
                check=True, capture_output=True, text=True,
            )
        actual = subprocess.check_output(
            ["git", "-C", str(target), "rev-parse", "HEAD"], text=True, stderr=subprocess.DEVNULL
        ).strip()
    except RuntimeError:
        raise
    except Exception:
        raise RuntimeError("Source checkout failed; check the repository URL and commit access.") from None
    if actual != commit:
        raise RuntimeError("Checked-out source SHA does not match the requested commit.")
    dirty = subprocess.check_output(
        ["git", "-C", str(target), "status", "--porcelain", "--untracked-files=all"], text=True
    )
    ignored_python = subprocess.check_output(
        ["git", "-C", str(target), "ls-files", "--others", "--ignored", "--exclude-standard", "-z"], text=True
    )
    importable_suffixes = (".py", ".pyc", ".pyo", ".so", ".pyd", ".dll", ".dylib")
    if dirty.strip() or any(path.lower().endswith(importable_suffixes) for path in ignored_python.split("\0") if path):
        raise RuntimeError("Source checkout is not clean; use a fresh Colab runtime.")
    if not (target / "ai/runtime/app.py").is_file():
        raise RuntimeError("The selected commit does not contain the AI Runtime source.")
    return target


REPO_URL = input("Repository URL: ").strip()
REPO_COMMIT = input("Immutable 40-character commit SHA: ").strip().lower()
repo_root = checkout_source(REPO_URL, REPO_COMMIT)
print(f"Source verified at {REPO_COMMIT}")

In [ ]:
# Environment: require one Colab T4 and preserve every already-installed package.
import importlib.metadata
import pathlib, re, subprocess, sys, tempfile

RUNTIME_PINS = {
    "fastapi": ("0.162.0", "fastapi==0.162.0"),
    "uvicorn": ("0.54.0", "uvicorn==0.54.0"),
    "jsonschema": ("4.26.0", "jsonschema[format]==4.26.0"),
    "transformers": ("5.21.0", "transformers==5.21.0"),
    "accelerate": ("1.15.0", "accelerate==1.15.0"),
    "lm-format-enforcer": ("0.11.3", "lm-format-enforcer==0.11.3"),
}


def _installed_package_versions():
    versions = {}
    for distribution in importlib.metadata.distributions():
        name = distribution.metadata.get("Name")
        if name:
            versions[re.sub(r"[-_.]+", "-", name).lower()] = distribution.version
    return versions


def _jsonschema_format_extras_missing(installed_versions):
    try:
        from packaging.requirements import Requirement
        requirements = importlib.metadata.requires("jsonschema") or []
        for text in requirements:
            requirement = Requirement(text)
            if requirement.marker and requirement.marker.evaluate({"extra": "format"}):
                name = re.sub(r"[-_.]+", "-", requirement.name).lower()
                version = installed_versions.get(name)
                if version is None or not requirement.specifier.contains(version, prereleases=True):
                    return True
        return False
    except Exception:
        return True


def dependency_plan(installed_versions, torch_version):
    if not torch_version:
        raise RuntimeError("Colab PyTorch is not installed; use a fresh Colab T4 runtime.")
    for name, (expected, _spec) in RUNTIME_PINS.items():
        actual = installed_versions.get(name)
        if actual is not None and actual != expected:
            raise RuntimeError(f"Installed {name} version is incompatible; use a fresh Colab runtime.")
    install = [spec for name, (version, spec) in RUNTIME_PINS.items() if installed_versions.get(name) is None]
    if installed_versions.get("jsonschema") == RUNTIME_PINS["jsonschema"][0] and _jsonschema_format_extras_missing(installed_versions):
        install.append(RUNTIME_PINS["jsonschema"][1])
    constraints = dict(installed_versions)
    constraints["torch"] = torch_version
    constraint_text = "\n".join(f"{name}=={version}" for name, version in sorted(constraints.items())) + "\n"
    return {"install": install, "constraints": constraint_text, "torch": torch_version}


def install_missing_runtime_dependencies():
    installed = _installed_package_versions()
    torch_before = installed.get("torch")
    plan = dependency_plan(installed, torch_before)
    if not plan["install"]:
        return installed
    constraint_path = None
    try:
        with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", prefix="ai-runtime-constraints-", delete=False) as stream:
            stream.write(plan["constraints"])
            constraint_path = stream.name
        command = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "-c", constraint_path]
        subprocess.run(command + ["--dry-run"] + plan["install"], check=True, capture_output=True, text=True)
        subprocess.run(command + plan["install"], check=True, capture_output=True, text=True)
    except Exception:
        raise RuntimeError("Missing runtime dependencies cannot be installed without changing existing packages; use a fresh Colab runtime.") from None
    finally:
        if constraint_path:
            pathlib.Path(constraint_path).unlink(missing_ok=True)
    after = _installed_package_versions()
    if after.get("torch") != torch_before or any(after.get(name) != version for name, version in installed.items()):
        raise RuntimeError("Dependency installation changed an existing package; restart with a fresh Colab runtime.")
    if any(after.get(name) != version for name, (version, _spec) in RUNTIME_PINS.items()):
        raise RuntimeError("Pinned AI Runtime dependencies are incomplete; restart with a fresh Colab runtime.")
    return after


def verify_colab_environment():
    try:
        devices = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=name,uuid,memory.total", "--format=csv,noheader"], text=True
        ).strip().splitlines()
    except Exception:
        raise RuntimeError("Colab GPU is unavailable; select a Tesla T4 runtime.") from None
    if len(devices) != 1 or "T4" not in devices[0]:
        raise RuntimeError(f"Expected one Tesla T4, found: {devices}")
    gpu_name, gpu_uuid, gpu_memory = [value.strip() for value in devices[0].split(",", 2)]
    torch_before = _installed_package_versions().get("torch")
    if not torch_before:
        raise RuntimeError("Colab's existing PyTorch is unavailable; use a fresh T4 runtime.")
    import torch
    if not torch.cuda.is_available() or torch.version.cuda is None:
        raise RuntimeError("Colab's existing PyTorch cannot access CUDA.")
    install_missing_runtime_dependencies()
    if torch.__version__ != torch_before or not torch.cuda.is_available():
        raise RuntimeError("The existing Colab PyTorch installation changed or lost CUDA support.")
    return {"gpu_name": gpu_name, "gpu_uuid": gpu_uuid, "gpu_memory": gpu_memory, "torch_version": torch_before}


environment = verify_colab_environment()
gpu_name = environment["gpu_name"]
gpu_uuid = environment["gpu_uuid"]
print(f"GPU: {gpu_name} ({environment['gpu_memory']}) | PyTorch {environment['torch_version']} | CUDA available")

In [ ]:
# Start: FastAPI startup completion follows the worker's READY handshake.
import getpass, os, pathlib, re, signal, subprocess, time
from urllib.request import urlopen

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_REVISION = "989aa7980e4cf806f80c7fef2b1adb7bc71aa306"
if "server" not in globals():
    server = None
if "log_file" not in globals():
    log_file = None
if "runtime_token" not in globals():
    runtime_token = None
log_path = pathlib.Path("/content/ai-runtime-serve/uvicorn.log")
log_path.parent.mkdir(parents=True, exist_ok=True)


def _log_text():
    if not log_path.exists():
        return ""
    return log_path.read_text(encoding="utf-8", errors="replace")


def recent_logs(limit=20):
    safe_lines = []
    for line in _log_text().splitlines():
        if (
            re.fullmatch(r"(?:INFO|WARNING|ERROR|CRITICAL): (?:Started server process \[\d+\]|Waiting for application startup|Application startup complete|Application startup failed\. Exiting\.|Shutting down|Finished server process \[\d+\])", line)
            or re.fullmatch(r"(?:INFO|WARNING|ERROR|CRITICAL): AI model loaded in [0-9]+(?:\.[0-9]+)? seconds", line)
            or re.fullmatch(r"(?:INFO|WARNING|ERROR|CRITICAL): Uvicorn running on http://127\.0\.0\.1:8000 \(Press CTRL\+C to quit\)", line)
        ):
            safe_lines.append(line)
    return "\n".join(safe_lines[-limit:])


def _signal_runtime_group(process, sig):
    try:
        os.killpg(process.pid, sig)  # Uvicorn starts a dedicated session; its worker inherits this process group.
    except ProcessLookupError:
        pass


def _shutdown_runtime_process(process):
    if process is None:
        return
    if process.poll() is None:
        process.terminate()
        try:
            process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            pass
    _signal_runtime_group(process, signal.SIGTERM)
    time.sleep(0.5)
    _signal_runtime_group(process, signal.SIGKILL)
    try:
        process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        pass


def _close_runtime_handles():
    global server, log_file, runtime_token
    server = None
    if log_file is not None and not log_file.closed:
        log_file.flush()
        log_file.close()
    log_file = None
    runtime_token = None


def start_runtime(repo_path, selected_gpu_uuid, token):
    global server, log_file, runtime_token
    if server is not None:
        if server.poll() is None:
            print("AI Runtime is already running; no model reload was started.")
            return {"status": "ALREADY_RUNNING"}
        _shutdown_runtime_process(server)
        _close_runtime_handles()
    if not token:
        raise ValueError("Enter an AI Runtime token.")
    runtime_token = token
    env = os.environ.copy()
    env.update({
        "AI_RUNTIME_TOKEN": token,
        "AI_RUNTIME_GPU_UUID": selected_gpu_uuid,
        "AI_RUNTIME_MODEL_ID": MODEL_ID,
        "AI_RUNTIME_MODEL_REVISION": MODEL_REVISION,
        "AI_RUNTIME_LOAD_MODE": "fp16",
        "PYTHONDONTWRITEBYTECODE": "1",
    })
    for name in ("AI_RUNTIME_LITE_DIAGNOSTICS", "AI_RUNTIME_LITE_DIAGNOSTICS_DIR", "AI_RUNTIME_BENCHMARK_MODE"):
        env.pop(name, None)
    log_file = open(log_path, "w", encoding="utf-8")
    try:
        server = subprocess.Popen(
            [sys.executable, "-m", "uvicorn", "ai.runtime.app:app", "--host", "127.0.0.1", "--port", "8000"],
            cwd=repo_path, env=env, stdout=log_file, stderr=subprocess.STDOUT, start_new_session=True,
        )
        del env
        deadline = time.monotonic() + 600
        while time.monotonic() < deadline:
            log_file.flush()
            text = _log_text()
            if "Application startup complete" in text and re.search(r"AI model loaded in [0-9.]+ seconds", text):
                seconds = re.findall(r"AI model loaded in ([0-9.]+) seconds", text)[-1]
                print(f"Worker/model READY at startup | {MODEL_ID}@{MODEL_REVISION} | FP16 | loaded in {seconds}s")
                print("Serving on Colab loopback 127.0.0.1:8000; run Status to check HTTP availability.")
                return {"status": "READY_AT_STARTUP", "model_load_seconds": float(seconds)}
            if server.poll() is not None:
                break
            time.sleep(2)
        safe_log = recent_logs()
        _shutdown_runtime_process(server)
        _close_runtime_handles()
        raise RuntimeError("FastAPI/worker did not reach READY. Recent runtime logs:\n" + safe_log)
    except Exception:
        if server is not None:
            _shutdown_runtime_process(server)
        _close_runtime_handles()
        raise


def stop_runtime():
    global server
    if server is None:
        print("AI Runtime is already stopped.")
        return
    process = server
    _shutdown_runtime_process(process)
    _close_runtime_handles()
    print("AI Runtime stopped; the Hugging Face cache was left intact.")


if server is not None and server.poll() is None:
    print("AI Runtime is already running; run Status to inspect it.")
else:
    runtime_token = getpass.getpass("AI Runtime token (hidden): ")
    if not runtime_token:
        raise ValueError("Enter an AI Runtime token.")
    start_runtime(repo_root, gpu_uuid, runtime_token)

In [ ]:
# Status checks HTTP availability and worker/model READY-at-startup separately.
def runtime_status():
    process_active = server is not None and server.poll() is None
    http_status = "NOT_AVAILABLE"
    if process_active:
        try:
            with urlopen("http://127.0.0.1:8000/openapi.json", timeout=2) as response:
                if response.status == 200:
                    http_status = "ACTIVE"
        except Exception:
            pass
    log = _log_text()
    ready_at_startup = (
        "Application startup complete" in log
        and re.search(r"AI model loaded in [0-9.]+ seconds", log) is not None
    )
    worker_model = "READY_AT_STARTUP" if ready_at_startup else "NOT_READY_OR_UNKNOWN"
    result = {"http": http_status, "worker_model": worker_model}
    print(f"HTTP server: {http_status}")
    print(f"Worker/model: {worker_model}")
    print("Worker/model status is startup-only; this runtime has no live worker health endpoint.")
    tail = recent_logs()
    if tail:
        print("\nRecent runtime logs:\n" + tail)
    return result


runtime_status()

In [ ]:
# Stop instructions. Run All intentionally leaves the server running.
print("When finished, explicitly call stop_runtime() in a new code cell. This cell does not stop the server.")